# M2-B2 — Anonymisation Franck

## Exploration des PII

Objectif de cette étape : charger `audit_sample.csv` (200 lignes stratifiées sur `income`) et analyser les types de PII présents dans `manager_comments` (noms, emails, téléphones, IBAN partiels, dates, lieux).

In [1]:
from pathlib import Path
import re

import pandas as pd

RANDOM_STATE = 42
DATA_DIR = Path("../data")
SAMPLE_PATH = DATA_DIR / "audit_sample.csv"

df = pd.read_csv(SAMPLE_PATH)
comments = df["manager_comments"].fillna("").astype(str)

display(pd.DataFrame({"rows": [len(df)], "columns": [df.shape[1]]}))
display(df[["income", "manager_comments"]].head(5))

,rows,columns
0,200,16


,income,manager_comments
0,<=50K,Daniel Murphy flagged a conflict with Gregory ...
1,<=50K,Dennis Huff requested a transfer. Approved by ...
2,<=50K,Strong year for Cynthia Hardy. Bonus wired to ...
3,>50K,Plan d'accompagnement ouvert pour Pierre Lapor...
4,>50K,Strong year for Matthew Acevedo. Bonus wired t...


In [4]:
# Verification de la stratification sur income : full dataset vs sample
FULL_PATH = DATA_DIR / "adult_income_with_comments.csv"
df_full = pd.read_csv(FULL_PATH)

income_compare = pd.concat(
    [
        df_full["income"].value_counts(normalize=True).rename("full_share"),
        df["income"].value_counts(normalize=True).rename("sample_share"),
    ],
    axis=1,
).fillna(0.0)

income_compare["abs_diff_pct_points"] = ((income_compare["sample_share"] - income_compare["full_share"]).abs() * 100).round(2)
income_compare = income_compare.assign(
    full_share=lambda d: (100 * d["full_share"]).round(2),
    sample_share=lambda d: (100 * d["sample_share"]).round(2),
)

display(income_compare)
display(pd.DataFrame({"max_abs_diff_pct_points": [income_compare["abs_diff_pct_points"].max()]}))

,full_share,sample_share,abs_diff_pct_points
income,,,
<=50K,75.92,50.0,25.92
>50K,24.08,50.0,25.92


,max_abs_diff_pct_points
0,25.92


In [2]:
# Regex heuristiques pour identifier les PII dans les commentaires
pattern_map = {
    "name": re.compile(r"\b[A-Z][a-z]+(?:[-'][A-Z][a-z]+)?\s[A-Z][a-z]+(?:[-'][A-Z][a-z]+)?\b"),
    "email": re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b"),
    "phone": re.compile(r"(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)\d{2,4}[\s.-]?\d{2,4}[\s.-]?\d{2,4}"),
    "iban_partial": re.compile(r"\b[A-Z]{2}\d{2}(?:[ -]?[A-Z0-9]{2,4}){2,}\b"),
    "date": re.compile(r"\b(?:\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{4}-\d{2}-\d{2})\b"),
}

# Lieux : on exploite les valeurs de native_country quand elles apparaissent dans les commentaires
countries = sorted(
    {str(c).strip() for c in df["native_country"].dropna().unique() if str(c).strip()},
    key=len,
    reverse=True,
 )
country_pattern = re.compile("|".join(re.escape(c) for c in countries)) if countries else None

records = []
for pii_type, pattern in pattern_map.items():
    row_count = comments.apply(lambda txt: bool(pattern.search(txt))).sum()
    total_matches = comments.apply(lambda txt: len(pattern.findall(txt))).sum()
    records.append({
        "pii_type": pii_type,
        "rows_with_pii": int(row_count),
        "total_matches": int(total_matches),
        "share_rows_pct": round(100 * row_count / len(comments), 2),
    })

if country_pattern is not None:
    location_rows = comments.apply(lambda txt: bool(country_pattern.search(txt))).sum()
    location_matches = comments.apply(lambda txt: len(country_pattern.findall(txt))).sum()
else:
    location_rows = 0
    location_matches = 0

records.append(
    {
        "pii_type": "location",
        "rows_with_pii": int(location_rows),
        "total_matches": int(location_matches),
        "share_rows_pct": round(100 * location_rows / len(comments), 2),
    }
)

pii_frequency = pd.DataFrame(records).sort_values("rows_with_pii", ascending=False).reset_index(drop=True)
display(pii_frequency)

,pii_type,rows_with_pii,total_matches,share_rows_pct
0,name,198,384,99.0
1,phone,119,146,59.5
2,date,86,86,43.0
3,email,85,85,42.5
4,iban_partial,0,0,0.0
5,location,0,0,0.0


In [5]:
# Exemples pour inspection qualitative (2 exemples par type)
def extract_examples(text_series: pd.Series, regex: re.Pattern, max_examples: int = 2) -> pd.DataFrame:
    matched = text_series[text_series.apply(lambda txt: bool(regex.search(txt)))].head(max_examples)
    if matched.empty:
        return pd.DataFrame({"example": ["Aucun exemple trouvé"]})
    return pd.DataFrame({"example": matched.values})

def show_examples(df_examples: pd.DataFrame) -> None:
    styled = df_examples.style.set_properties(
        subset=["example"],
        **{"white-space": "pre-wrap", "text-align": "left"},
    )
    display(styled)

with pd.option_context("display.max_colwidth", None):
    for pii_type, pattern in pattern_map.items():
        display(pd.DataFrame({"pii_type": [pii_type]}))
        show_examples(extract_examples(comments, pattern, max_examples=2))

    display(pd.DataFrame({"pii_type": ["location"]}))
    if country_pattern is not None:
        show_examples(extract_examples(comments, country_pattern, max_examples=2))
    else:
        show_examples(pd.DataFrame({"example": ["Aucun pays exploitable dans native_country"]}))

,pii_type
0,name


,example
0,Daniel Murphy flagged a conflict with Gregory Meyer. Mediation scheduled. HR follow-up: ashleyraymond@example.org.
1,Dennis Huff requested a transfer. Approved by Courtney Miller. Reach them at 310.727.4959 for handover. Ticket HR-24680.


,pii_type
0,email


,example
0,Daniel Murphy flagged a conflict with Gregory Meyer. Mediation scheduled. HR follow-up: ashleyraymond@example.org.
1,Plan d'accompagnement ouvert pour Pierre Laporte. Coaching confié à Pauline Monnier (rcordova@example.net).


,pii_type
0,phone


...

,example
0,Aucun exemple trouvé


,pii_type
0,date


,example
0,Strong year for Cynthia Hardy. Bonus wired to account ****1400. Validated by Brittany Garcia on 2025-02-11.
1,Strong year for Matthew Acevedo. Bonus wired to account ****8655. Validated by David Parrish on 2025-08-03.


,pii_type
0,location


,example
0,Aucun exemple trouvé


## Mise en place spaCy NER

Objectif : charger `en_core_web_md`, extraire `PERSON`, `GPE`, `ORG` + `EMAIL` (regex complémentaire), puis vérifier qualitativement ce que le modèle capte ou rate sur ~10 exemples.

In [12]:
import spacy

# Chargement du modele EN (fallback telechargement si absent)
try:
    nlp_en = spacy.load("en_core_web_md")
except OSError:
    from spacy.cli import download

    download("en_core_web_md")
    nlp_en = spacy.load("en_core_web_md")

email_regex = re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b")
fr_name_regex = re.compile(
    r"\b(?:[A-Z][a-z]+(?:[-'][A-Z][a-z]+)?\s){1,2}[A-Z][a-z]+(?:[-'][A-Z][a-z]+)?\b"
)

# Heuristique FR plus stricte pour eviter les faux positifs massifs
fr_markers = [
    "plan d'accompagnement",
    "alerte comportementale",
    "entretien annuel",
    "suivi rh",
    "joignable au",
    "confié à",
    "signalée par",
    "référent",
    "bon élément",
 ]
accent_regex = re.compile(r"[àâçéèêëîïôûùüÿœ]", flags=re.IGNORECASE)

def looks_french(text: str) -> bool:
    txt = text.lower()
    has_fr_phrase = any(marker in txt for marker in fr_markers)
    has_accent = bool(accent_regex.search(txt))
    return has_fr_phrase or has_accent

def extract_entities(text: str) -> dict:
    doc = nlp_en(text)
    persons = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    gpes = [ent.text for ent in doc.ents if ent.label_ == "GPE"]
    orgs = [ent.text for ent in doc.ents if ent.label_ == "ORG"]
    emails = email_regex.findall(text)

    return {
        "PERSON_spacy": sorted(set(persons)),
        "GPE_spacy": sorted(set(gpes)),
        "ORG_spacy": sorted(set(orgs)),
        "EMAIL_regex": sorted(set(emails)),
        "is_likely_french": looks_french(text),
        "PERSON_regex_fr_hint": sorted(set(fr_name_regex.findall(text))),
    }

In [10]:
# Extraction NER sur tout le sample
ner_rows = []
for idx, text in comments.items():
    ents = extract_entities(text)
    ner_rows.append(
        {
            "row_id": int(idx),
            "comment": text,
            **ents,
        }
    )

ner_df = pd.DataFrame(ner_rows)

# Frequencedetection par type
freq_summary = pd.DataFrame(
    {
        "entity_type": ["PERSON_spacy", "GPE_spacy", "ORG_spacy", "EMAIL_regex"],
        "rows_detected": [
            int(ner_df["PERSON_spacy"].apply(len).gt(0).sum()),
            int(ner_df["GPE_spacy"].apply(len).gt(0).sum()),
            int(ner_df["ORG_spacy"].apply(len).gt(0).sum()),
            int(ner_df["EMAIL_regex"].apply(len).gt(0).sum()),
        ],
    }
)
freq_summary["share_rows_pct"] = (100 * freq_summary["rows_detected"] / len(ner_df)).round(2)
display(freq_summary.sort_values("rows_detected", ascending=False))

# Indicateur de trous potentiels sur les commentaires FR
fr_subset = ner_df[ner_df["is_likely_french"]].copy()
if len(fr_subset) > 0:
    fr_gap = pd.DataFrame(
        {
            "n_likely_fr_comments": [len(fr_subset)],
            "fr_with_PERSON_spacy": [int(fr_subset["PERSON_spacy"].apply(len).gt(0).sum())],
            "fr_with_PERSON_regex_hint": [int(fr_subset["PERSON_regex_fr_hint"].apply(len).gt(0).sum())],
        }
)
    display(fr_gap)
else:
    display(pd.DataFrame({"info": ["Aucun commentaire detecte comme potentiellement FR par heuristique"]}))

,entity_type,rows_detected,share_rows_pct
0,PERSON_spacy,200,100.0
3,EMAIL_regex,85,42.5
2,ORG_spacy,39,19.5
1,GPE_spacy,3,1.5


,n_likely_fr_comments,fr_with_PERSON_spacy,fr_with_PERSON_regex_hint
0,23,23,21


In [11]:
# Verification qualitative sur ~10 exemples
quali_cols = [
    "row_id",
    "is_likely_french",
    "comment",
    "PERSON_spacy",
    "PERSON_regex_fr_hint",
    "EMAIL_regex",
    "GPE_spacy",
    "ORG_spacy",
]

# 5 exemples FR probables + 5 exemples random pour comparer
fr_examples = ner_df[ner_df["is_likely_french"]].head(5)
random_examples = ner_df.sample(5, random_state=RANDOM_STATE)
quali_10 = pd.concat([fr_examples, random_examples], axis=0).drop_duplicates(subset=["row_id"]).head(10)

with pd.option_context("display.max_colwidth", None):
    display(quali_10[quali_cols])

,row_id,is_likely_french,comment,PERSON_spacy,PERSON_regex_fr_hint,EMAIL_regex,GPE_spacy,ORG_spacy
3,3,True,Plan d'accompagnement ouvert pour Pierre Laporte. Coaching confié à Pauline Monnier (rcordova@example.net).,"[Pauline Monnier, Pierre Laporte]","[Pauline Monnier, Pierre Laporte]",[rcordova@example.net],[],[]
8,8,True,Alerte comportementale signalée par Valentine Ribeiro au sujet de Noël Joubert de Lagarde. Suivi RH : harpermonica@example.net.,"[Joubert de Lagarde, Valentine Ribeiro au sujet de Noël, comportementale signalée]",[Valentine Ribeiro],[harpermonica@example.net],[],[]
9,9,True,Prime versée à Paulette Gosselin-Delahaye sur le compte ****9722. Contrôle : Odette Thomas.,"[Odette Thomas, Paulette Gosselin-Delahaye, versée]","[Odette Thomas, Paulette Gosselin-Delahaye]",[],[],[]
11,11,True,Alerte comportementale signalée par Jacqueline Noël-Buisson au sujet de Sabine Rey du Grondin. Suivi RH : cody76@example.com.,"[Jacqueline Noël-Buisson au sujet de Sabine Rey du Grondin, comportementale signalée]",[Sabine Rey],[cody76@example.com],[],[]
12,12,True,RAS pour Julien Mace cette année. Manager : Philippine de la Duval.,[Julien Mace],[Julien Mace],[],[],[RAS]
95,95,True,"Entretien annuel de Gabriel Navarro : bon élément. Référent : Patrick-Luc Turpin, joignable au 756.173.8012.","[Gabriel Navarro, Patrick-Luc Turpin]","[Gabriel Navarro, Patrick-Luc Turpin]",[],[],[]
15,15,False,"Annual review for Charlene Sanders: solid contributor, no concern. Manager: Jose Baker, contact kerri41@example.org.","[Charlene Sanders, Jose Baker]","[Charlene Sanders, Jose Baker]",[kerri41@example.org],[],[kerri41@example.org]
30,30,False,"Jesse Kelley is a strong promotion candidate this year. Discussed with HR (Beverly Gibson, 811.888.1248). Budget pre-approved on account ****5829.","[Beverly Gibson, Jesse Kelley]","[Beverly Gibson, Jesse Kelley]",[],[],[]
158,158,False,Behavioral concern raised by colleague Tina Dickerson. Manager Roger Montes (carlosthompson@example.com) to follow up by 2025-07-01. Reference ticket HR-58698.,"[Roger Montes, Tina Dickerson]","[Manager Roger Montes, Tina Dickerson]",[carlosthompson@example.com],[],[]
128,128,False,Sean Gonzalez requested a transfer. Approved by Daniel Clark. Reach them at 810.780.1639 for handover. Ticket HR-85089.,"[Daniel Clark, Sean Gonzalez]","[Daniel Clark, Sean Gonzalez]",[],[],[]


### Limite documentée (point clé reflexion.md)

Le modèle `en_core_web_md` capte correctement une grande partie des entités sur commentaires en anglais, mais il peut rater des noms dans les commentaires francophones (~12 % du corpus attendu).

Mesure de mitigation proposée :
- garder un complément regex pour `EMAIL` `PHONE` et `DATE` et un **hint regex** pour noms multi-mots ;
- détecter les commentaires FR puis appliquer `fr_core_news_md` en second passage ;
- documenter explicitement ce trou de couverture dans `reflexion.md` au lieu de l'ignorer.